# Lucknow air quality: cleaning the data (2020 to 2025)

This notebook takes your raw files and turns them into **one clean table**, one row per hour, with pollution and weather side by side.

**Before you start**, put these files in the **same folder as this notebook** (or upload them next to it):
1. The 8 pollution files from CPCB, named like `Raw_Data_2019_site_277_lalbagh_lucknow_cpcb_1Hr.csv` (2019 to 2026)
2. The ERA5 weather file, named like `reanalysis-era5-single-levels-timeseries-....csv` (unzip the ERA5 zip first)

Then run the cells **from top to bottom** (Shift + Enter on each). Every step prints something so you can see what happened.

## Step 1: Load the tools

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Step 2: Load the pollution files
Each file is one year. We find all 8, stick them together, and use the time column as the row label.

In [ ]:
files = sorted(glob.glob("*Raw_Data_*lalbagh*1Hr.csv"))
print(len(files), "pollution files found (should be 8)")
for f in files:
    print("  ", f)

df = pd.concat([pd.read_csv(f, na_values=["NA"], parse_dates=["Timestamp"]) for f in files])
df = df.sort_values("Timestamp").set_index("Timestamp")
print("Rows:", len(df))
df.head()

**What to expect:** 8 files and 67,920 rows (every hour from 1 Jan 2019 to 30 Sep 2026).

If it says `0 files found`, the files are not in the same folder as this notebook.

## Step 3: Keep only the columns we need
The raw files have 25 columns. Most are empty or unreliable (for example the weather sensors at the station change between years). We keep six pollutants and give them short names.

In [ ]:
df = df.rename(columns={
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "SO2 (µg/m³)": "so2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3",
})
df = df[["pm25", "pm10", "no2", "so2", "co", "o3"]]
df = df.asfreq("h")          # make sure every hour has a row, even if it is empty
df.index.name = "time"

print("Percent of hours with no reading, per column:")
print((df.isna().mean() * 100).round(1))

## Step 4: Clean the pollution data
Sensors sometimes give impossible readings. We turn these into blanks (`NaN`) rather than guessing a value. Each rule prints how many hours it changed.

| Rule | Why |
|---|---|
| PM2.5 or PM10 that is zero or negative | cannot be real |
| PM2.5 or PM10 of 995 or more | the instrument maxes out at about 1000; that is the machine's limit, not the air |
| CO of 9.99 or more | the CO column is capped at exactly 10 |
| Same PM value for 6 hours in a row | the sensor is stuck |

In [ ]:
# Rule 1: zero or negative PM
for col in ["pm25", "pm10"]:
    bad = df[col] <= 0
    print(f"{col}: zero or negative -> {bad.sum()} hours removed")
    df.loc[bad, col] = np.nan

# Rule 2: instrument ceiling
for col in ["pm25", "pm10"]:
    bad = df[col] >= 995
    print(f"{col}: at the ceiling (>= 995) -> {bad.sum()} hours removed")
    df.loc[bad, col] = np.nan

bad = df["co"] >= 9.99
print(f"co: at the ceiling (>= 9.99) -> {bad.sum()} hours removed")
df.loc[bad, "co"] = np.nan

# Rule 3: stuck sensor (same value 6 or more hours in a row)
def stuck(series, n=6):
    run_id = (series != series.shift()).cumsum()                 # new number every time the value changes
    run_length = series.groupby(run_id).transform("size")        # how long each run is
    return (run_length >= n) & series.notna()

for col in ["pm25", "pm10"]:
    bad = stuck(df[col])
    print(f"{col}: stuck sensor -> {bad.sum()} hours removed")
    df.loc[bad, col] = np.nan

**What to expect:** PM2.5 loses only about 30 hours out of 67,920, so cleaning is very light. That is good: it means the raw data was mostly fine.

### Two warning flags (we mark these, we do NOT delete them)
Real events such as Diwali can look like errors, so we keep the values and just add a True/False column.

In [ ]:
df["pm25_spike"] = df["pm25"].diff().abs() > 300       # jumped by more than 300 in one hour
df["pm25_above_pm10"] = df["pm25"] > df["pm10"]        # impossible: small particles cannot outweigh all particles

print("Spike hours flagged:", df["pm25_spike"].sum())
print("PM2.5 greater than PM10 hours flagged:", df["pm25_above_pm10"].sum())

## Step 5: Load the weather file (ERA5)
This file is hourly weather for the Lucknow area. It has no gaps. We load it and convert the units into ones that make sense.

In [ ]:
era_files = glob.glob("*reanalysis-era5*.csv")
print("Weather files found:", era_files)

era = pd.concat([pd.read_csv(f, parse_dates=["valid_time"]) for f in era_files])
era = era.drop_duplicates("valid_time").set_index("valid_time").sort_index()

# unit conversions
era["temp_c"] = era["t2m"] - 273.15                     # kelvin -> celsius
dew_c = era["d2m"] - 273.15
era["humidity_pct"] = 100 * np.exp(17.625 * dew_c / (243.04 + dew_c)) / np.exp(17.625 * era["temp_c"] / (243.04 + era["temp_c"]))
era["wind_speed"] = np.hypot(era["u10"], era["v10"])    # wind speed from its two directions
era["blh_m"] = era["blh"]                               # boundary layer height in metres
era["pressure_hpa"] = era["sp"] / 100                   # pascals -> hPa
era["rain_mm"] = era["tp"] * 1000                       # metres -> millimetres

weather = era[["temp_c", "humidity_pct", "wind_speed", "blh_m", "pressure_hpa", "rain_mm"]]
print("Weather covers", weather.index.min(), "to", weather.index.max())
weather.head()

## Step 6: Fix the time zone (important!)
The weather file uses **UTC**, but the pollution data uses **Indian time (IST)**, which is 5 hours 30 minutes ahead. If we skip this, the weather would be matched to the wrong hours.

Adding 5h30m puts every weather reading at half past the hour, so we average the two neighbouring readings to get a value exactly on the hour.

In [ ]:
weather = weather.copy()
weather.index = weather.index + pd.Timedelta(hours=5, minutes=30)

whole_hours = pd.date_range(weather.index.min().ceil("h"), weather.index.max().floor("h"), freq="h")
weather = weather.reindex(weather.index.union(whole_hours)).interpolate("time", limit=1).reindex(whole_hours)
weather.index.name = "time"

weather.head()

## Step 7: Put pollution and weather in one table
We join them by time, add year / month / hour columns, and keep only **2020 to 2025** (the years where we have both).

In [ ]:
data = df.join(weather, how="left")

data["year"] = data.index.year
data["month"] = data.index.month
data["hour"] = data.index.hour

data = data[(data["year"] >= 2020) & (data["year"] <= 2025)]
data = data.dropna(subset=["temp_c"])        # drop the first few hours of 2020 that have no weather yet

print("Final table:", data.shape[0], "rows,", data.shape[1], "columns")
data.head()

**What to expect:** about 52,600 rows.

## Step 8: Check that it looks right
Always look at the result before using it. First, how complete is each year?

In [ ]:
print("Percent of hours with a valid PM2.5 reading, by year:")
print((data["pm25"].notna().groupby(data["year"]).mean() * 100).round(1))

Second, a picture. Pollution should be **high in winter (Nov to Feb) and low in the monsoon (Jul to Sep)**, every year. If the picture shows that, the cleaning and the dates are working.

In [ ]:
monthly = data["pm25"].resample("MS").mean()

plt.figure(figsize=(12, 4))
monthly.plot()
plt.ylabel("PM2.5 (µg/m³)")
plt.title("Lucknow (Lalbagh): monthly average PM2.5")
plt.grid(alpha=0.3)
plt.show()

And a first look at the idea behind the whole project: **when the boundary layer is low (air is trapped near the ground), is pollution higher?**

In [ ]:
winter = data[data["month"].isin([11, 12, 1])].dropna(subset=["pm25", "blh_m"])
groups = pd.qcut(winter["blh_m"], 4, labels=["lowest", "low-mid", "high-mid", "highest"])
print("Average winter PM2.5 by boundary layer height (lowest = most trapped air):")
print(winter.groupby(groups, observed=True)["pm25"].mean().round(0))

## Step 9: Save the clean table
This writes a CSV file next to the notebook. From here you can build your model, dashboard, or Power BI report from this one file.

In [ ]:
data.to_csv("lucknow_clean_2020_2025.csv")
print("Saved: lucknow_clean_2020_2025.csv")